# CNN+LSTM Arabic ASR
Train a compact CNN+LSTM CTC model on a small Common Voice Arabic subset.
Edit paths and limits below before running.

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader

sys.path.append(os.path.abspath(".."))

from cnn_lstm.dataset import ASRDataset, collate_fn
from cnn_lstm.model import CNNLSTM
from cnn_lstm.utils import CharacterEncoder, calculate_wer, greedy_decoder

In [ ]:
DATA_DIR = "../data/raw"
SAVE_DIR = "../saved_models"
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-3
LIMIT = 500
DEV_LIMIT = 100

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
device

In [ ]:
clips_dir = os.path.join(DATA_DIR, "clips")
train_tsv = os.path.join(DATA_DIR, "train.tsv")
dev_tsv = os.path.join(DATA_DIR, "dev.tsv")

char_encoder = CharacterEncoder()
train_data = ASRDataset(train_tsv, clips_dir, limit=LIMIT)
dev_data = ASRDataset(dev_tsv, clips_dir, limit=DEV_LIMIT)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
dev_loader = DataLoader(
    dev_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

len(train_data), len(dev_data)

In [ ]:
model = CNNLSTM(input_dim=80, num_classes=char_encoder.vocab_size).to(device)
criterion = torch.nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def compute_ctc_loss(outputs, labels, pooled_lengths, lbl_lens):
    outputs = outputs.permute(1, 0, 2)
    if device.type == "mps":
        return criterion(
            outputs.to("cpu"),
            labels.to("cpu"),
            pooled_lengths.to("cpu"),
            lbl_lens.to("cpu"),
        )
    return criterion(outputs, labels, pooled_lengths, lbl_lens)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    steps = 0

    for batch in train_loader:
        if batch is None:
            continue
        specs, labels, in_lens, lbl_lens = batch
        pooled_lengths = in_lens // 4
        valid_mask = pooled_lengths >= lbl_lens
        if not torch.any(valid_mask):
            continue
        specs = specs[valid_mask].to(device)
        labels = labels[valid_mask].to(device)
        pooled_lengths = pooled_lengths[valid_mask]
        lbl_lens = lbl_lens[valid_mask]

        optimizer.zero_grad()
        outputs = model(specs)
        loss = compute_ctc_loss(outputs, labels, pooled_lengths, lbl_lens)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    avg_loss = total_loss / max(steps, 1)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {avg_loss:.4f}")

os.makedirs(SAVE_DIR, exist_ok=True)
model_path = os.path.join(SAVE_DIR, "asr_model.pth")
torch.save(model.state_dict(), model_path)
model_path

In [ ]:
model.eval()
all_preds, all_truths = [], []

with torch.no_grad():
    for batch in dev_loader:
        if batch is None:
            continue
        specs, labels, _, lbl_lens = batch
        specs = specs.to(device)
        outputs = model(specs)
        preds = greedy_decoder(outputs, char_encoder)
        all_preds.extend(preds)
        for label, length in zip(labels, lbl_lens):
            all_truths.append(char_encoder.decode(label[:length].tolist()))

total_wer = sum(calculate_wer(t, p) for t, p in zip(all_truths, all_preds) if t)
avg_wer = total_wer / max(len(all_truths), 1)
avg_wer

In [ ]:
sample = None
for i in range(len(dev_data)):
    item = dev_data[i]
    if item is not None:
        sample = item
        break

if sample is None:
    raise RuntimeError("No valid samples found in dev set.")

spec, _ = sample
spec = spec.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(spec)
    pred = greedy_decoder(output, char_encoder)[0]

pred